# 00. Setup and Import Existing Models
Load the required libraries and set up the environment.
Additionally, download Glove vectors if not already present.


In [46]:
# Importing necessary libraries
import os
import sys
import pickle
from urllib.request import urlopen, urlretrieve
import zipfile
import deepl
import numpy as np

# Read environment variables
DEEPL_API_KEY = os.getenv("DEEPL_API_KEY")

# Clients
deepl_client = deepl.DeepLClient(DEEPL_API_KEY)

# External links
url_glove = "https://nlp.stanford.edu/data/glove.6B.zip"
urls_chisco = open("chisco_pkl_urls.txt").read().splitlines()

In [47]:
def get_glove_embeddings(glove_url):
    # Download from https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip using os process
    urlretrieve(glove_url, "glove.zip")
    with zipfile.ZipFile("glove.zip", "r") as zip_ref:
        zip_ref.extractall(".")

    path_to_glove_file = "glove.6B.100d.txt"
    embeddings_index = {}
    with __builtins__.open(path_to_glove_file) as f:
        for line in f:
            word, coefs = line.split(maxsplit=1)
            coefs = np.fromstring(coefs, "f", sep=" ")
            embeddings_index[word] = coefs
    return embeddings_index

glove_embeddings = get_glove_embeddings(url_glove)

# 01. Data Download and Preprocessing
Data is obtained from the [Chisco Dataset](https://www.nature.com/articles/s41597-024-04114-1). For performance reasons, only the .pkl files are downloaded.

In [37]:
def preprocess_picke_from_url(url):
    """
    Function to parse pickle file from a given URL.
    """
    try:
        with urlopen(url) as response:
            obj = pickle.load(response)
            for p in obj:
                p["translated_text"] = deepl_client.translate_text(p["text"], target_lang="EN-US").text
                p["eeg_features"] = p["input_features"]
                p["embeddings"] = np.random.rand(768)  # Placeholder for actual embedding retrieval
                p["input_features"] = []    
                del p["input_features"]
            return obj
    except Exception as e:
        print(f"Error loading pickle file from {url}: {e}")
        return None

In [ ]:
pickles = [
    preprocess_picke_from_url(url) for url in urls_chisco
]

In [39]:
pickles[0][0]

{'text': '加拿大真是个好地方',
 'translated_text': 'Canada is such a great place.',
 'eeg_features': array([[[ 3.50555641e-06,  3.68708718e-06,  1.60666774e-06, ...,
          -1.30720178e-07, -1.70827514e-06, -1.97441746e-06],
         [ 3.87624597e-06,  3.80952214e-06,  6.78143691e-07, ...,
           3.13501876e-06,  2.87287327e-07,  6.80169929e-07],
         [ 4.42657917e-07,  6.82549039e-07, -1.40955460e-07, ...,
           2.74492406e-06,  2.17777650e-06,  1.14774375e-06],
         ...,
         [-1.89642970e+01, -1.96243672e+01, -2.01101912e+01, ...,
          -8.35112671e+00, -8.38873182e+00, -9.86580351e+00],
         [ 2.06813083e+01,  1.83533653e+01,  1.75513076e+01, ...,
           1.80840216e+01,  1.67494934e+01,  1.42919978e+01],
         [ 6.52800000e+04,  6.52800000e+04,  6.52800000e+04, ...,
           6.52800000e+04,  6.52800000e+04,  6.52800000e+04]]],
       shape=(1, 125, 1651)),
 'embeddings': array([2.08042797e-01, 7.71673143e-01, 8.75746542e-01, 5.35603977e-01,
        1